# Subh-e-Ziraat — Sindh Water & Hydrology Dataset (1981–2025)
**MUET Final Year Project · district × year features keyed on `['District', 'Year']`**

This notebook has two data layers:

| Layer | Source | How it is obtained |
|---|---|---|
| **A. Soil-water & hydrology** (`GWETROOT`, `GWETTOP`, `EVPTRN`, `PRECTOTCORR`) | NASA POWER monthly API (MERRA-2 land model) | **Live HTTP download** in this notebook, cached to `water_cache/` |
| **B. Canal discharge & groundwater depth** | IRSA / SIDA / PCRWR / WAPDA publications | **You supply the official files** (PDF / Excel / CSV) in `raw_water_docs/`; the notebook extracts, cleans and standardises them |

**On data authenticity.** Nothing in this notebook generates, simulates or fills in values. Layer A is downloaded live. For layer B, no official source offers a stable public API for Sindh canal or groundwater history, so the code cannot download it for you. If you provide no files, those columns are simply absent from the output (never invented).

**Be precise about what Layer A is:** NASA POWER soil wetness and evapotranspiration are **MERRA-2 reanalysis model outputs on a ~50 km grid**, not in-situ measurements. That is a standard, citable product for crop modelling, but describe it as *modelled* in your report.

**Year convention** (same as your crop data, `"1981-82"` → `1981`): Kharif = May–Oct of *Y*; Rabi = Nov of *Y* → Apr of *Y+1*; Annual = Jan–Dec of *Y*.

## 0. Setup

In [ ]:
# pip install requests pandas numpy pdfplumber openpyxl      # run once if needed
import os, re, json, time, glob, calendar
import numpy as np
import pandas as pd
import requests

try:
    import pdfplumber
except ImportError:
    pdfplumber = None
    print("pdfplumber not installed -> `pip install pdfplumber` is needed only for PDF ingestion (section 3).")

START_YEAR      = 1981
END_YEAR_CROP   = 2025                  # last crop-season start year you need
END_YEAR_API    = END_YEAR_CROP + 1     # Rabi 2025 ends Apr 2026, so download one more year

CACHE_WATER = "water_cache"             # raw NASA JSON, one file per district
RAW_DIR     = "raw_water_docs"          # YOUR official files go here (section 3)
EXTRACT_DIR = "extracted_tables"        # tables pulled out of PDFs, for inspection

for d in [CACHE_WATER, RAW_DIR, f"{RAW_DIR}/canal", f"{RAW_DIR}/groundwater", EXTRACT_DIR]:
    os.makedirs(d, exist_ok=True)

EXPECTED_MONTHS = {"Annual": 12, "Kharif": 6, "Rabi": 6}

## 1. Districts, coordinates and name normalisation
Approximate district headquarters. **Edit spellings so they match `df_crop['District']` exactly.** Aliases are matched after lower-casing, removing brackets, dots, hyphens and the word "district", so `"Nawabshah (Shaheed Benazirabad)"` and `"Kambar-Shahdadkot District"` also resolve.

In [ ]:
SINDH_DISTRICTS = {   # District: (latitude, longitude)
    "Badin": (24.655, 68.837),               "Dadu": (26.730, 67.775),
    "Ghotki": (28.006, 69.316),              "Hyderabad": (25.396, 68.373),
    "Jacobabad": (28.281, 68.438),           "Jamshoro": (25.430, 68.280),
    "Kambar Shahdadkot": (27.588, 67.998),   "Karachi": (24.861, 67.010),
    "Kashmore": (28.433, 69.583),            "Khairpur": (27.529, 68.759),
    "Larkana": (27.559, 68.212),             "Matiari": (25.597, 68.447),
    "Mirpurkhas": (25.526, 69.015),          "Naushahro Feroze": (26.840, 68.123),
    "Sanghar": (26.046, 68.949),             "Shaheed Benazirabad": (26.245, 68.410),
    "Shikarpur": (27.955, 68.638),           "Sujawal": (24.607, 68.071),
    "Sukkur": (27.705, 68.857),              "Tando Allahyar": (25.463, 68.718),
    "Tando Muhammad Khan": (25.123, 68.536), "Tharparkar": (24.737, 69.797),
    "Thatta": (24.747, 67.924),              "Umerkot": (25.361, 69.736),
}

# keys are in *cleaned* form (see _clean_key): lower-case, no dots/hyphens/brackets
DISTRICT_ALIASES = {
    "nawabshah": "Shaheed Benazirabad", "benazirabad": "Shaheed Benazirabad",
    "shaheed benazir abad": "Shaheed Benazirabad", "sba": "Shaheed Benazirabad",
    "naushero feroze": "Naushahro Feroze", "naushahro firoz": "Naushahro Feroze",
    "nowshero feroze": "Naushahro Feroze", "naushero firoz": "Naushahro Feroze",
    "mirpur khas": "Mirpurkhas",
    "tharparker": "Tharparkar", "thar parkar": "Tharparkar", "thar": "Tharparkar",
    "tando muhammad khan": "Tando Muhammad Khan", "tando m khan": "Tando Muhammad Khan",
    "tm khan": "Tando Muhammad Khan",
    "tando allah yar": "Tando Allahyar", "t allahyar": "Tando Allahyar",
    "kambar": "Kambar Shahdadkot", "kamber shahdadkot": "Kambar Shahdadkot",
    "qambar shahdadkot": "Kambar Shahdadkot", "qambar": "Kambar Shahdadkot",
    "umer kot": "Umerkot", "thatha": "Thatta", "tatta": "Thatta",
    "kashmor": "Kashmore", "shikarpoor": "Shikarpur",
}
_CANON = {k.lower(): k for k in SINDH_DISTRICTS}

def _clean_key(x):
    s = str(x).lower()
    s = re.sub(r"\(.*?\)", " ", s)          # drop bracketed text
    s = re.sub(r"[_\-/]", " ", s).replace(".", "")
    s = re.sub(r"\bdistrict\b", " ", s)
    return " ".join(s.split())

def normalize_district(name):
    if pd.isna(name):
        return name
    key = _clean_key(name)
    if key in _CANON:
        return _CANON[key]
    return DISTRICT_ALIASES.get(key, " ".join(str(name).split()))   # unknown -> only trimmed

print(len(SINDH_DISTRICTS), "districts configured")

## 2. Layer A — NASA POWER soil-water & hydrology (live download)

Endpoint: `https://power.larc.nasa.gov/api/temporal/monthly/point`, community `AG`.

| Parameter | Meaning | Note |
|---|---|---|
| `GWETROOT` | Root-zone soil wetness, 0–1 | dimensionless; MERRA-2 root zone ≈ top 1 m |
| `GWETTOP` | Surface soil wetness, 0–1 | dimensionless; top few cm |
| `EVPTRN` | Evapotranspiration **energy flux** | In the AG community the API reports this as **MJ/m²/day**, not mm. The notebook **reads the unit from the live response** and converts to mm/day only if it is an energy unit |
| `PRECTOTCORR` | Bias-corrected precipitation, mm/day | converted to mm per month, then summed |

In [ ]:
POWER_URL    = "https://power.larc.nasa.gov/api/temporal/monthly/point"
WATER_PARAMS = ["GWETROOT", "GWETTOP", "EVPTRN", "PRECTOTCORR"]
LATENT_HEAT  = 2.45          # MJ per kg of evaporated water (~2.45 at 20 °C): 1 kg/m2 == 1 mm

def fetch_power_water(district, lat, lon, retries=4):
    # Download (or load from cache) the monthly NASA POWER JSON for one district.
    cache_file = os.path.join(CACHE_WATER, district.replace(" ", "_") + ".json")
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    params = {
        "parameters": ",".join(WATER_PARAMS), "community": "AG",
        "latitude": lat, "longitude": lon,
        "start": START_YEAR, "end": END_YEAR_API, "format": "JSON",
    }
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(POWER_URL, params=params, timeout=(15, 180))   # (connect, read)
            if 400 <= r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"HTTP {r.status_code} (not retrying): {r.text[:400]}")
            r.raise_for_status()
            js = r.json()
            par = js["properties"]["parameter"]
            missing = [p for p in WATER_PARAMS if p not in par]
            if missing:
                raise KeyError(f"parameters missing from response: {missing}")
            with open(cache_file, "w") as f:
                json.dump(js, f)
            return js
        except RuntimeError:
            raise
        except Exception as e:
            last_err = e
            wait = 5 * attempt
            print(f"  [{district}] attempt {attempt}/{retries} failed: {e} -> retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"NASA POWER download failed for {district}: {last_err}")

def tidy_power_water(js, district):
    par = js["properties"]["parameter"]
    df = pd.DataFrame({p: par[p] for p in WATER_PARAMS})
    df.index.name = "yyyymm"
    df = df.reset_index()
    df["cal_year"] = df["yyyymm"].str[:4].astype(int)
    df["month"] = df["yyyymm"].str[4:].astype(int)
    df = df[df["month"].between(1, 12)].drop(columns="yyyymm")     # month 13 = NASA annual mean
    df[WATER_PARAMS] = df[WATER_PARAMS].replace(-999, np.nan)       # NASA fill value
    df.insert(0, "District", district)
    return df

monthly_frames, UNITS = [], {}
for i, (dist, (lat, lon)) in enumerate(SINDH_DISTRICTS.items(), 1):
    print(f"[{i}/{len(SINDH_DISTRICTS)}] {dist} ({lat}, {lon})")
    js = fetch_power_water(dist, lat, lon)
    monthly_frames.append(tidy_power_water(js, dist))
    for p, meta in js.get("parameters", {}).items():      # units reported by the live API
        UNITS.setdefault(p, meta.get("units"))

df_water_monthly = pd.concat(monthly_frames, ignore_index=True)
print("\nUnits reported by NASA POWER:", {k: UNITS.get(k) for k in WATER_PARAMS})
print(df_water_monthly.shape)
df_water_monthly.head()

### 2b. Seasonal aggregation (Kharif / Rabi / Annual)

- `GWETROOT`, `GWETTOP`: **mean** over the window.
- `EVPTRN`: **mean** in the API's own units. If those are MJ/m²/day, a total in mm (`ET_mm`) is added using ET[mm] = EVPTRN ÷ 2.45 × days.
- `PRECIP_mm`: **sum** of monthly totals.
- `WaterBalance_mm = PRECIP_mm − ET_mm`. This is rainfall minus *modelled actual* ET, a rough indicator of how much the season depends on irrigation. It is **not** a crop water requirement.
- Windows with any missing month give NaN (nothing is silently averaged over a partial window).

In [ ]:
def expand_windows(df, year_col="cal_year", month_col="month"):
    # Tag every monthly row with the Annual / Kharif / Rabi window(s) it belongs to.
    ann = df.assign(Year=df[year_col], Season="Annual")
    khr = df[df[month_col].between(5, 10)].assign(Year=lambda x: x[year_col], Season="Kharif")
    rb = df[(df[month_col] >= 11) | (df[month_col] <= 4)].copy()
    rb["Year"] = np.where(rb[month_col] >= 11, rb[year_col], rb[year_col] - 1)   # Rabi belongs to start year
    rb["Season"] = "Rabi"
    return pd.concat([ann, khr, rb], ignore_index=True)

def build_nasa_water_features(df_m):
    d = df_m.copy()
    d["days"] = [calendar.monthrange(y, m)[1] for y, m in zip(d["cal_year"], d["month"])]
    d["PRECIP_mm"] = d["PRECTOTCORR"] * d["days"]

    et_unit = (UNITS.get("EVPTRN") or "")
    has_et_mm = "mj" in et_unit.lower()
    if has_et_mm:
        d["ET_mm"] = d["EVPTRN"] / LATENT_HEAT * d["days"]
    else:
        print(f"EVPTRN unit is '{et_unit}' -> no mm conversion applied; only the raw mean is kept.")

    long = expand_windows(d)
    grp = long.groupby(["District", "Year", "Season"])
    spec = {"GWETROOT": "mean", "GWETTOP": "mean", "EVPTRN": "mean", "PRECIP_mm": "sum"}
    if has_et_mm:
        spec["ET_mm"] = "sum"
    agg = grp.agg(spec)
    n_months = grp["month"].count()
    expected = agg.index.get_level_values("Season").map(EXPECTED_MONTHS).to_numpy()

    for v in spec:                                   # require every month of the window to be valid
        agg[v] = agg[v].where(grp[v].count().to_numpy() == expected)
    agg = agg[n_months.to_numpy() == expected]       # drop incomplete windows (e.g. 2026)

    if has_et_mm:
        agg["WaterBalance_mm"] = agg["PRECIP_mm"] - agg["ET_mm"]

    wide = agg.reset_index().pivot(index=["District", "Year"], columns="Season")
    wide.columns = [f"{season}_{var}" for var, season in wide.columns]
    wide = wide.reset_index()
    wide = wide[(wide["Year"] >= START_YEAR) & (wide["Year"] <= END_YEAR_CROP)]
    return wide

df_nasa_water = build_nasa_water_features(df_water_monthly)
NASA_FEATURES = [c for c in df_nasa_water.columns if c not in ("District", "Year")]
print(df_nasa_water.shape, "| years", df_nasa_water.Year.min(), "-", df_nasa_water.Year.max())
df_nasa_water.head()

## 3. Layer B — Canal irrigation & groundwater ingestion (IRSA / SIDA / PCRWR / WAPDA)

**Workflow**
1. Put the official files (PDF, `.xlsx`, `.xls` or `.csv`) in `raw_water_docs/canal/` and `raw_water_docs/groundwater/`.
2. Run **3a** (writes empty header-only templates) and **3b** (extracts PDF tables and shows every file's columns).
3. Fill in the column maps in **3c** so the file's headings map to the standard names below.
4. Run **3d**, which standardises the files and builds the season features.

**Standard schema the loader expects (after your column map is applied)**

| Dataset | Columns |
|---|---|
| Canal | `Entity` (canal, barrage **or** district) · `Year` · `Month` *or* `Season` *or* neither (= annual) · `Discharge_cusecs` and/or `Volume_MAF` |
| Groundwater | `District` · `Year` · `Month` *or* `Season` · `GW_Depth_m` *or* `GW_Depth_ft` (depth to water table) |

- `Year` may be `1981`, `1981-82` or a full date. `Month` may be `1–12`, `Jan`, `January`, or you can give a `Date` column.
- If you supply `Season`, accepted values are `Annual`, `Kharif`, `Rabi`, `PreMonsoon`, `PostMonsoon`. For those rows, `Year` must already be the **season start year**. Note IRSA's official Kharif/Rabi periods (Apr–Sep / Oct–Mar) differ from the May–Oct / Nov–Apr windows used here. Monthly rows are re-windowed to match your crop calendar; pre-labelled seasonal rows are used as given, so keep the label (`Kharif_…` columns will then mean IRSA's Kharif).
- **Canals do not coincide with districts.** For rows whose `Entity` is a canal or barrage, fill `canal_district_map.csv` (`Canal, District, Weight`). Flow is split across districts by `Weight` (normalised per canal; equal split if left blank). Use command-area proportions from SIDA/IRSA documents. **Nothing is pre-filled**, because I will not guess your allocations.

In [ ]:
# 3a. Header-only templates (no data rows). Only written if they don't already exist.
TEMPLATES = {
    "canal_template.csv":              ["Entity", "Year", "Month", "Discharge_cusecs", "Volume_MAF"],
    "groundwater_template.csv":        ["District", "Year", "Month", "GW_Depth_m"],
    "canal_district_map_template.csv": ["Canal", "District", "Weight"],
}
for fname, cols in TEMPLATES.items():
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path):
        pd.DataFrame(columns=cols).to_csv(path, index=False)
print("Templates available in", RAW_DIR, "->", list(TEMPLATES))

In [ ]:
# 3b. Readers: CSV / Excel / PDF -> DataFrame, plus numeric cleaning
SUPPORTED = (".csv", ".xlsx", ".xls", ".pdf")

def _dedupe(cols):
    seen, out = {}, []
    for c in cols:
        c = re.sub(r"\s+", " ", str(c).replace("\n", " ")).strip() or "col"
        seen[c] = seen.get(c, 0) + 1
        out.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
    return out

def extract_pdf_tables(pdf_path):
    # Extract every table from a text-based PDF with pdfplumber.
    # Scanned (image-only) PDFs contain no text layer and need OCR first (e.g. ocrmypdf).
    if pdfplumber is None:
        raise ImportError("pip install pdfplumber")
    frames = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, 1):
            tables = page.extract_tables()                       # ruled tables
            if not tables:                                       # fallback: whitespace-aligned tables
                tables = page.extract_tables({"vertical_strategy": "text", "horizontal_strategy": "text"})
            for tbl in tables:
                if not tbl or len(tbl) < 2:
                    continue
                df = pd.DataFrame(tbl[1:], columns=_dedupe([c if c else "" for c in tbl[0]]))
                df["_pdf_page"] = pno
                frames.append(df)
    if not frames:
        print(f"  ! no tables found in {os.path.basename(pdf_path)} (scanned image PDF? needs OCR)")
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    hdr = [c for c in out.columns if c != "_pdf_page"]
    out = out[~out[hdr].apply(lambda r: list(r.astype(str)) == hdr, axis=1)]   # repeated header rows
    return out.reset_index(drop=True)

def read_source_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        try:
            return pd.read_csv(path)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1")
    if ext in (".xlsx", ".xls"):
        sheets = pd.read_excel(path, sheet_name=None)
        return pd.concat([s.assign(_sheet=n) for n, s in sheets.items()], ignore_index=True)
    if ext == ".pdf":
        df = extract_pdf_tables(path)
        if len(df):
            out = os.path.join(EXTRACT_DIR, os.path.splitext(os.path.basename(path))[0] + "_raw.csv")
            df.to_csv(out, index=False)
        return df
    raise ValueError(f"unsupported file type: {path}")

def clean_numeric(s):
    # '1,234 cusecs', '12.5*', '-', 'N/A' -> float / NaN.  NOTE: commas are treated as THOUSANDS separators,
    # so a European-style decimal comma ('7,5') would be misread as 75 - fix such files before loading.
    if s.dtype.kind in "if":
        return s.astype(float)
    t = s.astype(str).str.strip().str.replace(",", "", regex=False)
    t = t.str.replace(r"[^0-9.\-eE+]", "", regex=True)
    t = t.replace({"": np.nan, "-": np.nan, ".": np.nan})
    return pd.to_numeric(t, errors="coerce")

def list_files(folder):
    return sorted(f for f in glob.glob(os.path.join(folder, "*"))
                  if f.lower().endswith(SUPPORTED) and "template" not in os.path.basename(f).lower())

# ---- inspection: shows the columns of every file so you can fill the maps in 3c ----
for kind in ("canal", "groundwater"):
    files = list_files(f"{RAW_DIR}/{kind}")
    print(f"\n=== {kind.upper()}: {len(files)} file(s) ===")
    for f in files:
        df = read_source_file(f)
        print(f"\n-- {os.path.basename(f)}  shape={df.shape}")
        print("   columns:", list(df.columns))
        print(df.head(3).to_string(max_colwidth=25))
if not any(list_files(f"{RAW_DIR}/{k}") for k in ("canal", "groundwater")):
    print("\nNo source files found yet. Add official IRSA/SIDA/PCRWR files to raw_water_docs/ and re-run.")

In [ ]:
# 3c. >>> EDIT ME <<<  Map each source file's headings to the standard names.
# Keys = headings exactly as printed by 3b; values = standard names (see the schema table above).
# Use per-file maps when files differ; a '*' entry applies to every file of that kind.
COLUMN_MAPS = {
    "canal": {
        # "*": {"Canal Name": "Entity", "Yr": "Year", "Mon": "Month", "Mean Discharge (Cusecs)": "Discharge_cusecs"},
        # "irsa_2010_report.pdf": {"Barrage/Canal": "Entity", "Flow": "Discharge_cusecs"},
    },
    "groundwater": {
        # "*": {"Dist": "District", "Year": "Year", "Month": "Month", "WTD (ft)": "GW_Depth_ft"},
    },
}
MIN_COVERAGE_MEAN = 0.75   # monthly rows: a window needs >=75% of its months for a mean-type metric (flows, depths)

In [ ]:
# 3d. Standardise -> allocate to districts -> seasonal features
MONTHS = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def parse_year(s):
    return pd.to_numeric(s.astype(str).str.extract(r"(\d{4})")[0], errors="coerce")

def parse_month(s):
    num = pd.to_numeric(s, errors="coerce")
    txt = s.astype(str).str.strip().str[:3].str.lower().map(MONTHS)
    return num.fillna(txt)

def load_standardised(kind, metric_cols):
    # Read every file of `kind`, apply its column map, return one standardised frame.
    frames = []
    for f in list_files(f"{RAW_DIR}/{kind}"):
        df = read_source_file(f)
        if df.empty:
            continue
        cmap = {**COLUMN_MAPS.get(kind, {}).get("*", {}), **COLUMN_MAPS.get(kind, {}).get(os.path.basename(f), {})}
        df = df.rename(columns=cmap)
        df["_source_file"] = os.path.basename(f)
        if "Date" in df.columns:
            dt = pd.to_datetime(df["Date"], errors="coerce")
            df["Year"] = df["Year"] if "Year" in df.columns else dt.dt.year
            df["Month"] = df["Month"] if "Month" in df.columns else dt.dt.month
        if "Entity" not in df.columns and "District" in df.columns:
            df["Entity"] = df["District"]
        if "Entity" not in df.columns or "Year" not in df.columns:
            print(f"  ! {os.path.basename(f)}: after mapping, 'Entity/District' and 'Year' are required -> skipped. "
                  f"Columns now: {list(df.columns)}")
            continue
        if "GW_Depth_ft" in df.columns and "GW_Depth_m" not in df.columns:
            df["GW_Depth_m"] = clean_numeric(df["GW_Depth_ft"]) * 0.3048
        present = [c for c in metric_cols if c in df.columns]
        if not present:
            print(f"  ! {os.path.basename(f)}: none of {metric_cols} found after mapping -> skipped.")
            continue
        for c in present:
            df[c] = clean_numeric(df[c])
        df["Year"] = parse_year(df["Year"])
        df["Month"] = parse_month(df["Month"]) if "Month" in df.columns else np.nan
        df["Season"] = df["Season"].astype(str).str.strip().str.title().replace({"Nan": np.nan, "": np.nan}) \
                       if "Season" in df.columns else np.nan
        df["Entity"] = df["Entity"].astype(str).str.strip()
        frames.append(df[["Entity", "Year", "Month", "Season", "_source_file"] + present])
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True).dropna(subset=["Year"])
    out["Year"] = out["Year"].astype(int)
    return out

def allocate_to_districts(df, metric_cols):
    # Rows already at district level keep weight 1; canal/barrage rows are split via canal_district_map.csv.
    df = df.copy()
    df["_norm"] = df["Entity"].map(normalize_district)
    is_dist = df["_norm"].isin(SINDH_DISTRICTS)
    direct = df[is_dist].assign(District=lambda x: x["_norm"])
    rest = df[~is_dist]
    parts = [direct]
    map_path = os.path.join(RAW_DIR, "canal_district_map.csv")
    if len(rest):
        if os.path.exists(map_path) and len(pd.read_csv(map_path)):
            cm = pd.read_csv(map_path)
            cm["_key"] = cm["Canal"].map(_clean_key)
            cm["District"] = cm["District"].map(normalize_district)
            cm["Weight"] = pd.to_numeric(cm.get("Weight"), errors="coerce")
            cm["Weight"] = cm["Weight"].fillna(1.0)
            cm["Weight"] = cm["Weight"] / cm.groupby("_key")["Weight"].transform("sum")
            rest = rest.assign(_key=rest["Entity"].map(_clean_key))
            m = rest.merge(cm[["_key", "District", "Weight"]], on="_key", how="left")
            unmapped = sorted(m.loc[m["District"].isna(), "Entity"].unique())
            if unmapped:
                print("  ! Entities with no district mapping (dropped):", unmapped)
            m = m.dropna(subset=["District"])
            for c in metric_cols:
                if c in m:
                    m[c] = m[c] * m["Weight"]
            parts.append(m.drop(columns=["_key", "Weight"]))
        else:
            print("  ! Non-district entities found but canal_district_map.csv is missing/empty "
                  "(copy canal_district_map_template.csv and fill it):", sorted(rest["Entity"].unique())[:10])
    out = pd.concat(parts, ignore_index=True)
    return out.drop(columns=["_norm"], errors="ignore")

def build_water_features(df, metric_aggs, combine, prefix_map):
    # df: District, Year, Month, Season, metrics (already at district level)
    # metric_aggs: {col: 'mean'|'sum'} aggregation over the months of a window
    # combine: how several entities/stations in one district-month are combined ('sum' for canals, 'mean' for wells)
    cols = list(metric_aggs)
    pieces = []

    # (1) monthly rows -> re-windowed to Annual/Kharif/Rabi with coverage rules
    mo = df[df["Month"].between(1, 12)].copy()
    if len(mo):
        mo["Month"] = mo["Month"].astype(int)
        dm = mo.groupby(["District", "Year", "Month"], as_index=False)[cols].agg(combine)
        dm = dm.rename(columns={"Year": "cal_year", "Month": "month"})
        long = expand_windows(dm)
        grp = long.groupby(["District", "Year", "Season"])
        res = grp.agg({c: metric_aggs[c] for c in cols})
        exp = res.index.get_level_values("Season").map(EXPECTED_MONTHS).to_numpy()
        for c in cols:
            need = np.ceil(MIN_COVERAGE_MEAN * exp) if metric_aggs[c] == "mean" else exp   # sums need full coverage
            res[c] = res[c].where(grp[c].count().to_numpy() >= need)
        pieces.append(res.reset_index())

    # (2) rows that carry their own Season label  (3) year-only rows -> Annual
    se = df[df["Month"].isna()].copy()
    se["Season"] = se["Season"].fillna("Annual")
    if len(se):
        pieces.append(se.groupby(["District", "Year", "Season"], as_index=False)[cols].agg(combine))

    if not pieces:
        return pd.DataFrame(columns=["District", "Year"])
    allp = pd.concat(pieces, ignore_index=True).drop_duplicates(["District", "Year", "Season"], keep="first")
    wide = allp.pivot(index=["District", "Year"], columns="Season")
    wide.columns = [f"{season}_{prefix_map.get(var, var)}" for var, season in wide.columns]
    wide = wide.reset_index()
    return wide[(wide["Year"] >= START_YEAR) & (wide["Year"] <= END_YEAR_CROP)]

# ---------------- canal ----------------
print("--- Canal ---")
canal_metrics = ["Discharge_cusecs", "Volume_MAF"]
raw_canal = load_standardised("canal", canal_metrics)
if len(raw_canal):
    used = [c for c in canal_metrics if c in raw_canal.columns and raw_canal[c].notna().any()]
    raw_canal = allocate_to_districts(raw_canal, used)
    aggs = {c: ("mean" if c == "Discharge_cusecs" else "sum") for c in used}
    df_canal = build_water_features(raw_canal, aggs, combine="sum",
                                    prefix_map={"Discharge_cusecs": "CanalDischarge_cusecs", "Volume_MAF": "CanalVolume_MAF"})
    print("canal features:", df_canal.shape)
else:
    df_canal = pd.DataFrame(columns=["District", "Year"])
    print("No canal data supplied -> canal columns will NOT be created.")

# ---------------- groundwater ----------------
print("--- Groundwater ---")
raw_gw = load_standardised("groundwater", ["GW_Depth_m"])
if len(raw_gw):
    raw_gw = allocate_to_districts(raw_gw, ["GW_Depth_m"])
    df_gw = build_water_features(raw_gw, {"GW_Depth_m": "mean"}, combine="mean", prefix_map={})
    print("groundwater features:", df_gw.shape)
else:
    df_gw = pd.DataFrame(columns=["District", "Year"])
    print("No groundwater data supplied -> groundwater columns will NOT be created.")

## 4. Assemble the hydrology table (`District`, `Year` primary key)

In [ ]:
df_hydrology = df_nasa_water.copy()
for part, flag in ((df_canal, "Canal_Data_Available"), (df_gw, "GW_Data_Available")):
    if len(part):
        part = part.copy()
        part[flag] = 1
        df_hydrology = df_hydrology.merge(part, on=["District", "Year"], how="left")
        df_hydrology[flag] = df_hydrology[flag].fillna(0).astype(int)

df_hydrology["Year"] = df_hydrology["Year"].astype(int)
df_hydrology = df_hydrology.sort_values(["District", "Year"]).reset_index(drop=True)

# integrity checks on the primary key
assert not df_hydrology.duplicated(["District", "Year"]).any(), "duplicate District-Year keys!"
assert set(df_hydrology["District"]) <= set(SINDH_DISTRICTS), "unexpected district spelling"

df_hydrology.to_csv("sindh_hydrology_1981_2025.csv", index=False)
print(df_hydrology.shape)
print(df_hydrology.dtypes.value_counts())
df_hydrology.head()

## 5. Merge with the crop dataset, clean, verify, export

Load your crop data first (uncomment the `read_csv` line if `df_crop` isn't already in memory).

**Missing-value policy**
- Rows missing any **NASA** feature are dropped (reported).
- Canal and groundwater columns are **left as NaN** where official data are unavailable, with `*_Data_Available` flags. Filling gaps with invented numbers would contaminate a project whose point is real data.
- If a model needs complete data, impute **inside your train/validation pipeline** (for example `SimpleImputer` fitted on training folds) to avoid leakage. XGBoost, LightGBM and HistGradientBoosting accept NaN natively.

In [ ]:
# df_crop = pd.read_csv("your_crop_dataset.csv")     # <- uncomment / adjust if needed
if "df_crop" not in globals():
    raise NameError("Load your crop dataframe as `df_crop` first (see the commented line above).")

df_crop = df_crop.copy()
df_crop["Year"] = pd.to_numeric(df_crop["Year"], errors="raise").astype(int)

before = df_crop["District"].copy()
df_crop["District"] = df_crop["District"].map(normalize_district)
changed = pd.DataFrame({"old": before, "new": df_crop["District"]}).query("old != new").drop_duplicates()
if len(changed):
    print("District names normalised:\n", changed.to_string(index=False))

unmatched = sorted(set(df_crop["District"]) - set(df_hydrology["District"]))
print("Crop districts without hydrology match:", unmatched or "none")

# ------------- the merge you specified -------------
df_merged = pd.merge(df_crop, df_hydrology, on=['District', 'Year'], how='inner')
# ----------------------------------------------------

print(f"crop rows {len(df_crop):,} -> merged rows {len(df_merged):,} "
      f"({len(df_crop) - len(df_merged):,} crop rows had no matching District-Year)")
assert len(df_merged) <= len(df_crop), "merge multiplied rows -> duplicate keys in df_hydrology"

In [ ]:
# ---- clean + verify ----
n0 = len(df_merged)
df_merged = df_merged.replace([np.inf, -np.inf], np.nan)

core = [c for c in NASA_FEATURES if c in df_merged.columns]          # mandatory (live NASA) features
df_merged = df_merged.dropna(subset=core).reset_index(drop=True)
print(f"dropped {n0 - len(df_merged):,} rows with missing NASA features")

# dtypes
df_merged["Year"] = df_merged["Year"].astype(int)
df_merged["District"] = df_merged["District"].astype(str)
feature_cols = [c for c in df_hydrology.columns if c not in ("District", "Year")]
for c in feature_cols:
    df_merged[c] = pd.to_numeric(df_merged[c], errors="coerce")

# sanity ranges (physical limits)
for c in [c for c in core if c.endswith(("GWETROOT", "GWETTOP"))]:
    assert df_merged[c].between(0, 1).all(), f"{c} outside 0-1"
assert (df_merged[[c for c in core if "PRECIP_mm" in c]] >= 0).all().all(), "negative precipitation"

# missing-value report (optional columns)
miss = df_merged[feature_cols].isna().mean().mul(100).round(1)
print("\n% missing per hydrology column (only non-zero shown):")
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")

OUT_FILE = "sindh_integrated_crop_water_dataset_1981_2025.csv"
df_merged.to_csv(OUT_FILE, index=False)
print(f"\nSaved {OUT_FILE}: {df_merged.shape[0]:,} rows x {df_merged.shape[1]} columns")
df_merged.head()

### Optional: combine with the climate + soil dataset from the previous notebook
If `sindh_climate_soil_1981_2025.csv` exists, this joins it to the merged table on `['District','Year']`. Overlapping columns (for example `*_PRECIP_mm`, present in both) are kept once, from this notebook.

In [ ]:
CLIMATE_FILE = "sindh_climate_soil_1981_2025.csv"
if os.path.exists(CLIMATE_FILE):
    df_cs = pd.read_csv(CLIMATE_FILE)
    df_cs["District"] = df_cs["District"].map(normalize_district)
    overlap = [c for c in df_cs.columns if c in df_merged.columns and c not in ("District", "Year")]
    df_all = pd.merge(df_merged, df_cs.drop(columns=overlap), on=["District", "Year"], how="left")
    print(f"added {df_cs.shape[1] - 2 - len(overlap)} climate/soil columns; dropped duplicates: {overlap}")
    df_all.to_csv("sindh_integrated_crop_climate_water_dataset.csv", index=False)
    print("Saved sindh_integrated_crop_climate_water_dataset.csv", df_all.shape)
else:
    print(CLIMATE_FILE, "not found - skipping (optional).")

## Notes for your report
- **Citation:** NASA POWER (data from NASA GMAO MERRA-2); cite the POWER referencing page and give your download date.
- **Soil wetness and ET are modelled**, representing ~50 km grid cells. Validate against any local observations you have (e.g. a PCRWR or SIDA site) before claiming field-scale accuracy.
- **EVPTRN is actual evapotranspiration from a land model**, so `WaterBalance_mm` mostly reflects irrigation dependence. The land model does not know about canal water, which is exactly why the canal-discharge layer matters in Sindh.
- **District boundaries changed** (Sujawal, Tando Muhammad Khan, Tando Allahyar, Matiari, Jamshoro, Kambar Shahdadkot, Umerkot, Shaheed Benazirabad). Hydrology is indexed by current district points; check how your crop rows label the older parent districts.
- **Canal-to-district allocation** is a modelling assumption. State the weights and their source in your methodology.